# nb04 — Resultados principais: a curva custo × recall (H0)

**O que este notebook faz:** lê o gold humano da T22, o judge congelado nas duas configurações e
os scores N1/N2 de três baterias, e produz as quatro figuras da T28:

1. a **curva custo × recall por camada** — `figures/fig05_custo_recall_h0.png`, a figura
   principal do trabalho;
2. o **recall por classe de falha (P, C, D)** — `figures/fig06_recall_por_classe_h0.png`, que é
   onde a predição de `ARQUITETURA §12` se verifica ou cai;
3. **`ΔRecall(N3 | N1+N2)` com IC bootstrap** — INS.2, o número que testa H0;
4. **H2, função contra argumentos** — `figures/fig07_h2_funcao_vs_args.png`, Δ pareado entre os
   dois modelos sobre a bateria principal (288 execuções);
5. **INS.9, a detecção de mutantes** — `figures/fig08_ins9_mutantes.png`, sobre a bateria de
   mutantes (150 execuções, com coluna de controle).

> **A aritmética não mora aqui.** Recall, falso alarme e o ganho incremental saem de
> `tapieval.scoring.ins`, que tem 16 testes. Um notebook que reimplementasse a conta produziria
> uma segunda versão dela, e a que aparece na figura seria justamente a que ninguém testou — é a
> mesma regra que o nb03 segue com o flip rate.

**O que este notebook NÃO mostra**, e cada limitação está repetida ao lado da figura que ela
afeta: na parte de H0, n = 20 execuções, a metade determinística do gold é a saída do próprio
detector (A27), e o gold foi rotulado às cegas — C2, C3 e C7 não existem nele. Nas partes de H2 e
INS.9, **nenhum número passa por N3**: as baterias foram pontuadas com o scorer determinístico
(`n1n2+taxonomia`), então o que se lê ali é o que a camada barata enxerga.

**O `pass^k` não está aqui** — é o nb05 (T29), que lê a mesma bateria principal.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
import sys

if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))

from tapieval import figuras as fg
from tapieval.labeling.cli import RotuloHumano
from tapieval.schema.trace import N3Judge
from tapieval.scoring import mutantes as mut
from tapieval.scoring.bateria import ler_scores, pontuar_bateria
from tapieval.scoring.comparacao import comparar_h2
from tapieval.scoring.ins import (
    CAMADAS,
    curva,
    falso_alarme,
    ganho_incremental,
    montar_item,
    recall_por_classe,
)

BATERIA = RAIZ / "runs" / "calibracao_2026-08-24"
PRINCIPAL = RAIZ / "runs" / "principal_2026_08" / "scores.jsonl"
MUTANTES = RAIZ / "runs" / "mutantes_2026_08" / "scores.jsonl"
JULGAMENTOS = RAIZ / "runs" / "judge_gold_calibracao_2026-08-24" / "julgamentos.jsonl"
ROTULOS = RAIZ / "labels" / "humano_2026-08-30.jsonl"
FIGURAS = RAIZ / "figures"

In [2]:
%load_ext watermark
%watermark -u -d -v -m -p pandas,plotly,kaleido

Last updated: 2026-09-01

Python implementation: CPython
Python version       : 3.14.7
IPython version      : 9.16.1

pandas : 3.0.5
plotly : 6.9.0
kaleido: 1.3.0

Compiler    : Clang 21.0.0 (clang-2100.1.1.101)
OS          : Darwin
Release     : 25.5.0
Machine     : arm64
Processor   : arm
CPU cores   : 10
Architecture: 64bit



---
## 1. O conjunto — e as três conferências que precedem qualquer número

O par que entra no recall é `(gold humano, detecção da camada)` **da mesma execução**. Três coisas
quebram isso sem quebrar nada:

- um rótulo da amostra de **melhoria** entrando no denominador (`METRICAS §5` proíbe: a fila é
  escolhida por dificuldade, e recall sobre casos difíceis não estima recall na população);
- uma execução julgada numa configuração e não na outra, que faria os dois pontos da curva terem
  denominadores diferentes;
- um rótulo apontando para um trace que não está nesta bateria — rotular contra uma bateria e
  julgar contra outra produz pares que nunca foram pares.

As três viram `assert` aqui, e não confiança em quem escreveu o notebook.

In [3]:
scores = {s.run_id: s for s in pontuar_bateria(BATERIA).scores}

rotulos = {}
for linha in ROTULOS.read_text(encoding="utf-8").splitlines():
    if not linha.strip():
        continue
    rotulo = RotuloHumano.model_validate_json(linha)
    if rotulo.amostra == "estimativa":
        rotulos[rotulo.run_id] = rotulo

julgamentos: dict[str, dict[str, N3Judge]] = {}
for linha in JULGAMENTOS.read_text(encoding="utf-8").splitlines():
    if not linha.strip():
        continue
    registro = json.loads(linha)
    if registro.get("erro"):
        continue
    julgamentos.setdefault(registro["run_id"], {})[registro["configuracao"]] = (
        N3Judge.model_validate(registro["julgamento"])
    )

sem_score = [r for r in rotulos if r not in scores]
assert not sem_score, f"rótulo aponta para trace fora desta bateria: {sem_score}"

incompletos = [r for r in rotulos if set(julgamentos.get(r, {})) != {"cego", "com_trace"}]
assert not incompletos, (
    f"{len(incompletos)} execução(ões) sem as duas configurações — os dois pontos da curva "
    f"teriam denominadores diferentes: {incompletos[:3]}"
)

itens = [
    montar_item(rid, scores[rid].n1, scores[rid].n2, rot.para_n4humano(), julgamentos[rid])
    for rid, rot in sorted(rotulos.items())
]
print(f"{len(itens)} execuções de estimativa · {sum(len(i.gold) for i in itens)} falhas no gold")

20 execuções de estimativa · 108 falhas no gold


---
## 2. O eixo x: custo por execução avaliada (INS.4)

O custo é medido em **tokens**, e não convertido em moeda. Preço de API muda, o número na figura
não deve mudar com ele — e a razão entre as camadas, que é o que a curva mostra, é a mesma nas
duas unidades. A camada determinística custa **zero token**: ela é função pura de
`(trace, gabarito)`, e `tests/test_repro.py` bloqueia `socket` para provar que continua sendo.

A mediana, e não a média: a distribuição de `tokens_in` do judge é assimétrica pelo tamanho da
evidência, e a média de um trace longo desloca a coluna inteira.

In [4]:
custos = {"n1n2": 0.0}
por_config: dict[str, list[float]] = {"cego": [], "com_trace": []}
for linha in JULGAMENTOS.read_text(encoding="utf-8").splitlines():
    if not linha.strip():
        continue
    registro = json.loads(linha)
    if registro.get("erro"):
        continue
    custo = registro["custo"]
    por_config[registro["configuracao"]].append(custo["tokens_in"] + custo["tokens_out"])

custos["n1n2n3_cego"] = float(pd.Series(por_config["cego"]).median())
custos["n1n2n3_com_trace"] = float(pd.Series(por_config["com_trace"]).median())

tabela = pd.DataFrame(
    [
        {
            "camada": r.camada,
            "recall": r.valor,
            "acertos": f"{r.n_acertos}/{r.n_gold}",
            "falso_alarme": falso_alarme(itens, r.camada),
            "tokens_por_execucao": custos[r.camada],
            "identidade_A27": r.identidade,
        }
        for r in curva(itens)
    ]
).set_index("camada")
tabela

,recall,acertos,falso_alarme,tokens_por_execucao,identidade_A27
camada,,,,,
n1n2,0.759259,82/108,0.000000,0.0,True
n1n2n3_cego,0.953704,103/108,0.009615,5015.0,False
n1n2n3_com_trace,0.953704,103/108,0.037383,8459.5,False


---
## 3. A figura principal — custo × recall

**Como ler.** Cada ponto é uma configuração do avaliador. Subir é detectar mais do gold; ir para a
direita é pagar mais por execução avaliada. A hipótese H0 prevê **retorno decrescente**: o
primeiro salto compra muito, o segundo compra pouco.

⚠️ **O primeiro ponto está marcado com contorno tracejado, e o motivo não é estético.** O recall
da camada determinística é **identidade, não medição** (A27): a metade P/D/C5 do gold é derivada
do mesmo `n1`/`n2` que a detecção. Lê-lo como "a camada barata já pega três quartos" é o erro que
o `METRICAS §11` existe para impedir. O que a figura mede de verdade é a **diferença** entre os
pontos — e é por isso que `METRICAS §7` marca a INS.2, e não a INS.1, como o número da hipótese.

In [5]:
TINTA, TINTA2, SUPERFICIE = fg.TINTA, fg.TINTA2, fg.SUPERFICIE
AZUL, AMBAR, VERDE, CINZA = fg.AZUL, fg.AMBAR, fg.VERDE, fg.CINZA

# O nome da camada é TICK do eixo x, e não anotação solta: anotação em `yref="paper"` cai
# em cima do título do eixo, e o overlap só aparece no PNG exportado — nunca no notebook.
ROTULO_DA_CAMADA = {
    "n1n2": "<b>N1+N2</b><br><sub>determinístico · 0 token</sub>",
    "n1n2n3_cego": "<b>+N3 cego</b><br><sub>judge vê a resposta</sub>",
    "n1n2n3_com_trace": "<b>+N3 com trace</b><br><sub>judge vê resposta e evidência</sub>",
}

x = [custos[c] for c in CAMADAS]
y = [tabela.loc[c, "recall"] for c in CAMADAS]

fig = go.Figure()
fig.add_scatter(
    x=x, y=y, mode="lines", line=dict(color=CINZA, width=3), showlegend=False, hoverinfo="skip"
)
# O ponto de identidade primeiro, com contorno tracejado; os dois medidos, sólidos.
fig.add_scatter(
    x=x[:1], y=y[:1], mode="markers+text", showlegend=False,
    marker=dict(size=20, color=SUPERFICIE, line=dict(color=TINTA2, width=3)),
    text=[f"  {y[0]:.1%}"], textposition="middle right",
    textfont=dict(color=TINTA2, size=13),
    hovertemplate="N1+N2 · recall %{y:.1%} · 0 token<extra></extra>",
)
fig.add_scatter(
    x=x[1:], y=y[1:], mode="markers+text", showlegend=False,
    marker=dict(size=20, color=AZUL, line=dict(color=AZUL, width=3)),
    text=[f"  {v:.1%}" for v in y[1:]], textposition="middle right",
    textfont=dict(color=AZUL, size=13),
    hovertemplate="recall %{y:.1%} · %{x:,.0f} tokens<extra></extra>",
)

fig.add_annotation(
    x=x[0], y=y[0], ax=30, ay=62, showarrow=True, arrowhead=0, arrowcolor=AMBAR, arrowwidth=2,
    text="<b>identidade, não medição</b><br><sub>A27 — a metade P/D do gold sai<br>do mesmo n1/n2 da detecção</sub>",
    font=dict(color=AMBAR, size=11), align="left", xanchor="left", yanchor="top",
)

ganho = ganho_incremental(itens)
fig.add_annotation(
    x=(x[0] + x[1]) / 2, y=(y[0] + y[1]) / 2, ax=60, ay=30, showarrow=True, arrowhead=0,
    arrowcolor=VERDE, arrowwidth=2, xanchor="left",
    text=(f"<b>ΔRecall = {ganho.delta:+.1%}</b><br>"
          f"<sub>IC95 [{ganho.ic95[0]:+.1%}, {ganho.ic95[1]:+.1%}] · INS.2</sub>"),
    font=dict(color=VERDE, size=12), align="left",
)

fig.update_layout(
    title=dict(
        text=("<b>O que cada camada de avaliação compra</b><br>"
              "<sub>recall contra o gold humano × tokens por execução avaliada · n = 20 · H0</sub>"),
        font=dict(color=TINTA, size=19), x=0, xanchor="left",
    ),
    xaxis=dict(
        title=dict(text="tokens por execução avaliada", standoff=26),
        gridcolor=CINZA, zeroline=False, range=[-900, max(x) * 1.30],
        tickmode="array", tickvals=x,
        ticktext=[ROTULO_DA_CAMADA[c] for c in CAMADAS],
    ),
    yaxis=dict(title="recall do gold", tickformat=".0%", gridcolor=CINZA,
               range=[0, 1.06], zeroline=False),
    plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE, font=dict(color=TINTA2, family=fg.FAMILIA),
    width=fg.LARGURA, height=580, margin=dict(l=80, r=50, t=95, b=120),
)
fg.exportar(fig, "fig05_custo_recall_h0", FIGURAS)
fig.show()

---
## 4. Onde o ganho está — o recall por classe de falha

Esta é a figura que decide a hipótese, e não a de cima. `ARQUITETURA §12` não prevê apenas que o
recall sobe: prevê **onde** ele sobe. *"P e D são detectáveis sem LLM, com custo perto de zero; C
exige N3."*

E o documento é explícito sobre a refutação: se o ganho aparecesse **em todas as classes por
igual**, a estratificação estaria errada e o achado seria sobre a taxonomia, não sobre as camadas.
A curva agregada da seção 3 não distingue os dois casos. Esta distingue.

In [6]:
por_classe = pd.DataFrame(
    {
        camada: {
            classe: r.valor for classe, r in recall_por_classe(itens, camada).items()
        }
        for camada in CAMADAS
    }
)
NOME_DA_CLASSE = {"P": "P — processo", "C": "C — conteúdo", "D": "D — decisão"}
tamanho_do_gold = {
    classe: recall_por_classe(itens, "n1n2")[classe].n_gold for classe in "PCD"
}

fig = go.Figure()
for camada, cor, nome in (
    ("n1n2", SUPERFICIE, "N1+N2 (determinístico)"),
    ("n1n2n3_cego", AZUL, "+N3 cego"),
):
    fig.add_bar(
        y=[NOME_DA_CLASSE[c] for c in "DPC"], x=[por_classe.loc[c, camada] for c in "DPC"],
        orientation="h", name=nome,
        marker=dict(color=cor, line=dict(color=TINTA2 if cor == SUPERFICIE else cor, width=2)),
        hovertemplate=nome + ": %{x:.0%}<extra></extra>",
    )

# O rótulo mostra ANTES → DEPOIS, e não só o delta. Em C o "antes" é 0%, e uma barra de
# comprimento zero é invisível: o leitor veria só a barra azul e concluiria que a camada
# determinística não foi medida ali — quando o que ela mediu foi exatamente zero, que é o
# achado. O número no texto é o que torna a barra ausente legível.
for i, classe in enumerate("DPC"):
    antes = por_classe.loc[classe, "n1n2"]
    depois = por_classe.loc[classe, "n1n2n3_cego"]
    fig.add_annotation(
        x=max(antes, depois) + 0.03, y=i, showarrow=False, xanchor="left",
        text=f"<b>{antes:.0%} → {depois:.0%}</b>"
        + f"<br><sub>{tamanho_do_gold[classe]} falhas no gold</sub>",
        font=dict(color=VERDE if depois > antes else CINZA, size=12), align="left",
    )

fig.update_layout(
    title=dict(
        text=("<b>O judge só compra conteúdo — processo e decisão o gabarito já dava</b><br>"
              "<sub>recall por classe de falha · n = 20 · a predição de H0, verificada<br>"
              "P e D em 100% são <b>identidade</b> (A27), não medição — o que se lê aqui é a "
              "linha do C</sub>"),
        font=dict(color=TINTA, size=19), x=0, xanchor="left",
    ),
    barmode="group", bargap=0.35,
    xaxis=dict(title="recall do gold", tickformat=".0%", gridcolor=CINZA, range=[0, 1.34]),
    yaxis=dict(title=""),
    legend=dict(orientation="h", y=-0.16, x=0),
    plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE, font=dict(color=TINTA2, family=fg.FAMILIA),
    width=fg.LARGURA, height=500, margin=dict(l=150, r=50, t=110, b=90),
)
fg.exportar(fig, "fig06_recall_por_classe_h0", FIGURAS)
fig.show()

---
## 5. INS.2 — o número, e o que ele não cobre

`ΔRecall(N3 | N1+N2)` é uma **diferença**, e é por isso que ele sobrevive à limitação do A27: a
parte determinística é idêntica nos dois lados e cancela, sobrando a fração do gold que só o judge
alcança.

O IC é bootstrap percentil sobre reamostragem **das execuções** — a unidade de observação
independente é a run, não o código. Reamostrar códigos trataria dois códigos da mesma execução
como duas medidas independentes e devolveria um intervalo estreito demais, ou seja: o erro sairia
na direção que faz o achado parecer mais firme do que é.

In [7]:
linhas = []
for de, para in (
    ("n1n2", "n1n2n3_cego"),
    ("n1n2n3_cego", "n1n2n3_com_trace"),
    ("n1n2", "n1n2n3_com_trace"),
):
    g = ganho_incremental(itens, de=de, para=para)
    linhas.append(
        {
            "de → para": f"{de} → {para}",
            "ΔRecall": g.delta,
            "IC95": f"[{g.ic95[0]:+.3f}, {g.ic95[1]:+.3f}]",
            "cruza zero?": "sim" if g.ic95[0] <= 0 <= g.ic95[1] else "não",
            "códigos ganhos": ", ".join(sorted(g.codigos_ganhos)) or "—",
        }
    )
pd.DataFrame(linhas).set_index("de → para")

,ΔRecall,IC95,cruza zero?,códigos ganhos
de → para,,,,
n1n2 → n1n2n3_cego,0.194444,"[+0.133, +0.248]",não,"C1, C4"
n1n2n3_cego → n1n2n3_com_trace,0.000000,"[-0.028, +0.028]",sim,C4
n1n2 → n1n2n3_com_trace,0.194444,"[+0.135, +0.248]",não,"C1, C4"


### O segundo ponto de N3 não paga — e a frase honesta não é a óbvia

`cego → com_trace` dá **Δ = 0,0** com o IC cruzando zero, enquanto o custo quase dobra e o falso
alarme sobe de 1,0% para 3,7%. A leitura tentadora é *"dar o trace ao judge não acrescenta nada"*.

⚠️ **Ela é falsa, e o motivo está no gold.** O rotulador humano trabalhou **às cegas** — a CLI de
rotulagem impõe isso por construção, porque o κ da INS.6 exige que humano e judge vejam o mesmo
insumo. Logo `afirmacoes_sem_suporte`, `contradiz_evidencia` e `recomendou_acao_sem_base` vêm
`None` no gold, e **C2, C3 e C7 não existem nele**. São exatamente os três códigos que só o judge
com trace pode detectar: o que ele achar ali entra como **falso alarme**, nunca como acerto.

A frase que os dados sustentam é: **o gold disponível não tem como dizer se o trace acrescenta.**
Medir isso exigiria uma segunda rotulagem humana, com evidência à vista — e aí o κ precisaria dos
dois conjuntos. Fica declarado como limitação e como trabalho futuro, com o custo já dimensionado.

In [8]:
resumo = {
    "n_execucoes": len(itens),
    "n_falhas_no_gold": sum(len(i.gold) for i in itens),
    "recall": {r.camada: round(r.valor, 4) for r in curva(itens)},
    "falso_alarme": {c: round(falso_alarme(itens, c), 4) for c in CAMADAS},
    "tokens_por_execucao": {c: round(custos[c], 1) for c in CAMADAS},
    "ins2_delta": round(ganho.delta, 4),
    "ins2_ic95": [round(v, 4) for v in ganho.ic95],
    "recall_por_classe": {
        camada: {classe: round(por_classe.loc[classe, camada], 4) for classe in "PCD"}
        for camada in CAMADAS
    },
}
(RAIZ / "docs" / "resultados_h0.json").write_text(
    json.dumps(resumo, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
print(json.dumps(resumo, indent=2, ensure_ascii=False))

{
  "n_execucoes": 20,
  "n_falhas_no_gold": 108,
  "recall": {
    "n1n2": 0.7593,
    "n1n2n3_cego": 0.9537,
    "n1n2n3_com_trace": 0.9537
  },
  "falso_alarme": {
    "n1n2": 0.0,
    "n1n2n3_cego": 0.0096,
    "n1n2n3_com_trace": 0.0374
  },
  "tokens_por_execucao": {
    "n1n2": 0.0,
    "n1n2n3_cego": 5015.0,
    "n1n2n3_com_trace": 8459.5
  },
  "ins2_delta": 0.1944,
  "ins2_ic95": [
    0.1346,
    0.2478
  ],
  "recall_por_classe": {
    "n1n2": {
      "P": 1.0,
      "C": 0.0,
      "D": 1.0
    },
    "n1n2n3_cego": {
      "P": 1.0,
      "C": 0.8077,
      "D": 1.0
    },
    "n1n2n3_com_trace": {
      "P": 1.0,
      "C": 0.8077,
      "D": 1.0
    }
  }
}


---
## 6. H2 — a diferença entre os modelos está nos argumentos, e não na função

`ARQUITETURA §12` prevê: *"comparados dois modelos locais de portes diferentes, a diferença será
maior na acurácia dos argumentos do que na escolha da função, porque escolher a tool é
reconhecimento de intenção e preencher argumentos exige leitura precisa do schema"*.

**A comparação é pareada por `(cenário, seed)`.** Os 18 cenários de test têm dificuldades muito
diferentes, e a variância entre eles é maior que a diferença entre os modelos: comparar
`média(14B)` com `média(8B)` põe essa variância inteira dentro do erro. O par cancela a
dificuldade, e o que sobra na diferença é a única coisa que variou — o modelo.

⚠️ **Duas execuções entram no fatorial e não entram na conta**, e a segunda muda a resposta:

| campo | valor de preenchimento | o que ele significa de verdade |
|---|---|---|
| `args_acc` | `0.0` quando `args_avaliados == 0` | acurácia condicional sobre conjunto vazio — **indefinida** |
| `decisao_correta` | `False` quando `decisao_prevista is None` | *"não houve decisão a comparar"*, não *"decidiu errado"* |

A segunda **não é simétrica entre os modelos**: das 37 execuções sem decisão, **30 são do 14B** —
é o X31, a taxa de `parse_erro` 15× maior do modelo maior. Contá-las como decisão errada importa
uma falha de **formato** para dentro de uma afirmação sobre **capacidade**, e o efeito é grande: o
Δ de `decisao_correta` vai de −0,102 para −0,139. O segundo número é, em boa parte, o X31
disfarçado de H2.

In [9]:
h2 = comparar_h2(ler_scores(PRINCIPAL), modelo_a="qwen3-8b", modelo_b="qwen3-14b")

pd.DataFrame(
    [
        {
            "métrica": d.metrica,
            "8B": d.media_a,
            "14B": d.media_b,
            "Δ (14B − 8B)": d.delta,
            "IC95": f"[{d.ic95[0]:+.3f}, {d.ic95[1]:+.3f}]",
            "n pares": d.n_pares,
            "descartados": d.n_descartados,
            "p_bootstrap": d.p_bootstrap,
            "leitura": d.leitura,
        }
        for d in h2
    ]
).set_index("métrica")

,8B,14B,Δ (14B − 8B),IC95,n pares,descartados,p_bootstrap,leitura
métrica,,,,,,,,
tool_f1,0.523062,0.543650,0.020588,"[-0.021, +0.062]",144,0,0.3310,cruza zero
tool_f1_liquido,0.383083,0.439982,0.056899,"[+0.008, +0.107]",144,0,0.0242,não cruza zero
args_acc,0.810397,0.748973,-0.061425,"[-0.123, +0.000]",124,21,0.0514,no limiar — este n não decide
decisao_correta,0.388889,0.287037,-0.101852,"[-0.204, +0.000]",108,37,0.0696,cruza zero


### O que a tabela responde, e a linha que ela se recusa a responder

**A magnitude de H2 se confirma; a direção cai.** A hipótese previa que a diferença fosse maior
nos argumentos que na função — e é (0,061 contra 0,021). Mas ela vinha com uma premissa implícita,
a de que o modelo maior ganharia nos dois eixos, e é a premissa que se quebra: o 14B escolhe
**melhor** a função e preenche **pior** os argumentos.

O ganho na função só aparece **depois de tirar o crédito da hidratação** (o X24): `tool_f1` bruto
cruza zero, `tool_f1_liquido` não. As duas tools que a hidratação chama por conta própria
atravessam a fronteira MCP e entram no F1 dos dois modelos por igual, diluindo a diferença — a
métrica líquida é a que mede o que o agente **escolheu** chamar.

⚠️ **`args_acc` está no limiar, e a tabela diz isso em vez de escolher um lado.** `p` = 0,0514
contra um corte de 0,05, com erro de Monte Carlo de 0,0022 — trocar a semente do bootstrap troca
o veredito. A primeira leitura destes dados, em 30/08, reportou *"não cruza zero"*; era a semente
falando. A frase que os dados sustentam é **"o efeito está no limiar e este n não decide"**, e é
por isso que `DiferencaPareada` carrega `p_bootstrap` ao lado do IC.

In [10]:
NOME_DA_METRICA = {
    "tool_f1": "<b>função</b><br><sub>N1.1 · tool_f1</sub>",
    "tool_f1_liquido": "<b>função (líquida)</b><br><sub>N1.1 sem hidratação · X24</sub>",
    "args_acc": "<b>argumentos</b><br><sub>N1.2 · condicional</sub>",
    "decisao_correta": "<b>decisão</b><br><sub>N1.4</sub>",
}
VERMELHO = fg.VERMELHO

fig = go.Figure()
fig.add_vline(x=0, line=dict(color=TINTA2, width=2, dash="dot"))
# De baixo para cima: a hipótese opõe função (em cima) a argumentos (embaixo).
for i, d in enumerate(reversed(h2)):
    # Âmbar quando o veredito é indeciso, azul quando ele se sustenta, cinza quando cruza zero.
    cor = AMBAR if not d.veredito_estavel else (AZUL if not d.cruza_zero else CINZA)
    fig.add_scatter(
        x=list(d.ic95), y=[i, i], mode="lines",
        line=dict(color=cor, width=6), showlegend=False, hoverinfo="skip",
    )
    fig.add_scatter(
        x=[d.delta], y=[i], mode="markers", showlegend=False,
        marker=dict(size=16, color=SUPERFICIE, line=dict(color=cor, width=4)),
        hovertemplate=f"{d.metrica}: %{{x:+.3f}}<extra></extra>",
    )
    # Os números vão para a MARGEM, e não para dentro do gráfico: com o eixo apertado em
    # ±0,25 (que é onde os intervalos vivem) qualquer rótulo interno cai em cima de um IC.
    fig.add_annotation(
        x=1.015, y=i, xref="paper", showarrow=False, xanchor="left",
        text=(f"<b>{d.delta:+.3f}</b>   <sub>IC [{d.ic95[0]:+.3f}, {d.ic95[1]:+.3f}]</sub><br>"
              f"<sub>n = {d.n_pares} pares · {d.leitura}</sub>"),
        font=dict(color=cor, size=12), align="left",
    )

fig.add_annotation(
    x=0, y=-0.30, xref="paper", yref="paper", showarrow=False, xanchor="left", align="left",
    text=("<b>argumentos está no limiar, e a figura diz isso em vez de escolher um lado:</b> "
          "p = 0,0514, a 0,0014 do corte de 0,05,<br>com erro de Monte Carlo de 0,0022. "
          "Trocar a semente do bootstrap troca o veredito — e nenhuma das duas frases é mais "
          "verdadeira."),
    font=dict(color=AMBAR, size=11.5),
)

fig.update_layout(
    title=dict(
        text=("<b>H2 se confirma na magnitude e cai na direção</b><br>"
              "<sub>Δ pareado (14B − 8B) por célula (mesmo cenário, mesma seed) · IC95 bootstrap "
              "sobre os pares<br>o modelo maior escolhe MELHOR a função e preenche PIOR os "
              "argumentos</sub>"),
        font=dict(color=TINTA, size=19), x=0, xanchor="left",
    ),
    xaxis=dict(title="Δ pareado (14B − 8B)", gridcolor=CINZA, zeroline=False,
               range=[-0.26, 0.15]),
    yaxis=dict(tickmode="array", tickvals=list(range(len(h2))),
               ticktext=[NOME_DA_METRICA[d.metrica] for d in reversed(h2)], showgrid=False),
    plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE, font=dict(color=TINTA2, family=fg.FAMILIA),
    width=fg.LARGURA, height=500, margin=dict(l=190, r=330, t=120, b=118),
)
fg.exportar(fig, "fig07_h2_funcao_vs_args", FIGURAS)
fig.show()

---
## 7. INS.9 — o instrumento distingue um agente sabotado de um agente são?

`METRICAS §7.1` degrada o agente de quatro maneiras deliberadas e pergunta se o avaliador
**percebe**. É poder de discriminação do instrumento, não desempenho do agente: um avaliador que
dá a mesma leitura para um agente são e um sabotado não mede nada, por mais bonita que seja a
métrica que imprime.

**A coluna `base` é o que torna a pergunta respondível.** A matriz original era `6 cenários × 4
mutantes × 5 seeds`, sem controle: a INS.9 seria *"fração distinguida do original"* **sem
original**. A `base` entrou em 30/08 com a margem que o corte do A16 liberou, e cada mutante é
comparado com a execução da **mesma célula** — mesmo cenário, mesma seed, com e sem sabotagem.

⚠️ **Distinguir não é detectar.** A fração de `METRICAS §7` é cega ao sinal: um par em que o
mutante fica com **menos** falha que a base conta como distinção exatamente como um em que ele
fica com mais. Por isso cada linha vem com a decomposição por direção, e com `poder_util` — a
fração distinguida **na direção esperada** — ao lado do valor.

In [11]:
pares_mut = mut.montar_pares(ler_scores(MUTANTES))

pd.DataFrame(
    [
        {
            "lente": d.lente,
            "variante": d.variante or "— agregada —",
            "INS.9": d.valor,
            "poder útil": d.poder_util,
            "invertida": d.fracao_invertida,
            "n pares": d.n_pares,
        }
        for d in mut.tabela(pares_mut)
    ]
).set_index(["lente", "variante"])

INS.9  poder útil  invertida  n pares
lente   variante                                              
codigos MUT1          0.866667    0.066667   0.166667       30
        MUT2          0.733333    0.166667   0.033333       30
        MUT3          1.000000    0.000000   0.733333       30
        MUT4          0.766667    0.066667   0.300000       30
        — agregada —  0.841667    0.075000   0.308333      120
sem_s2  MUT1          0.466667    0.266667   0.200000       30
        MUT2          0.300000    0.133333   0.166667       30
        MUT3          0.733333    0.000000   0.733333       30
        MUT4          0.300000    0.166667   0.133333       30
        — agregada —  0.450000    0.141667   0.308333      120
nominal MUT1          0.000000    0.000000   0.000000       30
        MUT2          0.000000    0.000000   0.000000       30
        MUT3          0.000000    0.000000   0.000000       30
        MUT4          0.000000    0.000000   0.000000       30
        — agregada —  0.000000    0.000000   0.000000      120

### Três lentes, três respostas — e a diferença entre elas é o achado

**Pela lente nominal o instrumento tem poder ZERO nos quatro mutantes.** Ele não distingue um
agente deliberadamente sabotado de um agente normal, porque os dois falham: o corte de
`METRICAS §6.5` exige trajetória perfeita, P1 dispara em quase toda run e vale S2. Não é falta de
detecção — é o **X33** medido onde ele mais dói, e é o argumento mais forte de que aquele corte
não pode ser a métrica reportada sozinha.

**E a variante "sem S2" não é a mitigação que o X33 supunha.** Ela discrimina (45% agregado contra
0%), mas 31% dos pares vão na direção **errada** — e o MUT3 é o caso extremo: `sucesso_binario_sem_s2`
dá **100% ao mutante e 27% à base**. Cortar o budget de 12 chamadas para 3 não melhora o agente;
melhora a nota.

**O mecanismo vale para a lente rica também, e é a limitação séria desta seção.** Vários códigos
da taxonomia são proporcionais à **oportunidade**: um agente que só pode dar três passos não tem
como acumular redundância, violar precedência ou perder cobertura em oito. No MUT3 o conjunto de
códigos do mutante é subconjunto **estrito** do da base em 22 dos 30 pares — 100% de distinção com
**0% de poder útil**. Agregado, o instrumento distingue 84% e acerta a direção em 8%.

Isso **não** diz que a taxonomia não presta: o MUT2 (conteúdo) tem a maior fração na direção certa,
e a maior parte das distinções é lateral — troca de falha, que é distinção legítima sem ser
confirmação. O que diz é que **INS.9 como fração é otimista por construção**, e que a linha que vai
para o README é o par `(valor, poder_util)`, nunca o primeiro sozinho.

In [12]:
ROTULO_MUT = {
    "MUT1": "MUT1<br><sub>tool some do<br>list_tools · P</sub>",
    "MUT2": "MUT2<br><sub>sem exigência<br>de citar · C</sub>",
    "MUT3": "MUT3<br><sub>budget 12 → 3<br>chamadas · P</sub>",
    "MUT4": "MUT4<br><sub>conclui sem<br>checar baseline · C</sub>",
}
LENTE = {
    "codigos": ("códigos da taxonomia", AZUL),
    "sem_s2": ("binário sem S2", AMBAR),
    "nominal": ("binário §6.5 (nominal)", VERMELHO),
}
DIRECAO = {
    "esperada": ("mutante piora — detecção", VERDE),
    "lateral": ("troca de falha — não confirma", CINZA),
    "invertida": ("mutante MELHORA a nota", VERMELHO),
}
MUTS = mut.variantes(pares_mut)

fig = make_subplots(
    rows=2, cols=1, row_heights=[0.46, 0.54], vertical_spacing=0.22,
    subplot_titles=(
        "<b>1 · INS.9 por lente</b>  <sub>— fração dos pares (base, mutante) em que a leitura "
        "muda</sub>",
        "<b>2 · a mesma fração, decomposta pela DIREÇÃO</b>  <sub>— lente dos códigos</sub>",
    ),
)
for lente, (nome, cor) in LENTE.items():
    valores = [mut.deteccao(pares_mut, lente, variante=m).valor for m in MUTS]
    fig.add_bar(
        x=MUTS, y=valores, name=nome, marker_color=cor, offsetgroup=lente,
        legendgroup="lente", legendgrouptitle_text="painel 1 — lente",
        text=[f"{v:.0%}" for v in valores], textposition="outside",
        textfont=dict(color=cor, size=12),
        hovertemplate=nome + ": %{y:.0%}<extra></extra>", row=1, col=1,
    )

# O painel 2 empilha; o painel 1 agrupa. Plotly tem UM `barmode` por figura, então o modo é
# `stack` e o painel 1 separa as barras por `offsetgroup` distinto — empilhar dentro de um
# grupo de um traço só é o mesmo que não empilhar.
for direcao, (nome, cor) in DIRECAO.items():
    por_mut = [mut.deteccao(pares_mut, "codigos", variante=m) for m in MUTS]
    valores = [dict(d.por_direcao).get(direcao, 0) / d.n_pares for d in por_mut]
    fig.add_bar(
        x=MUTS, y=valores, name=nome, marker_color=cor, offsetgroup="direcao",
        legendgroup="direcao", legendgrouptitle_text="painel 2 — direção",
        text=[f"{v:.0%}" if v >= 0.06 else "" for v in valores], textposition="inside",
        insidetextfont=dict(color=SUPERFICIE, size=12),
        hovertemplate=nome + ": %{y:.0%}<extra></extra>", row=2, col=1,
    )

fig.update_layout(
    title=dict(
        text=("<b>O instrumento distingue 84% dos mutantes — e em 31% deles a sabotagem MELHORA "
              "a nota</b><br><sub>INS.9 sobre 120 pares (6 cenários dev × 4 mutantes × 5 seeds), "
              "cada mutante contra a base da MESMA célula · poder útil agregado: <b>8%</b></sub>"),
        font=dict(color=TINTA, size=18), x=0, xanchor="left",
    ),
    barmode="stack", bargap=0.34,
    plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE, font=dict(color=TINTA2, family=fg.FAMILIA),
    width=fg.LARGURA, height=880, margin=dict(l=80, r=40, t=120, b=280),
    legend=dict(orientation="h", y=-0.13, x=0, font=dict(size=11), tracegroupgap=30),
)
fig.update_yaxes(tickformat=".0%", gridcolor=CINZA, range=[0, 1.17], row=1, col=1,
                 title=dict(text="fração distinguida", standoff=12))
fig.update_yaxes(tickformat=".0%", gridcolor=CINZA, range=[0, 1.0], row=2, col=1,
                 title=dict(text="fração dos pares", standoff=12))
fig.update_xaxes(tickmode="array", tickvals=MUTS,
                 ticktext=[ROTULO_MUT[m] for m in MUTS], showgrid=False)
# Os títulos dos painéis nascem centrados; o resto da figura é alinhado à esquerda.
for anotacao in fig.layout.annotations:
    anotacao.update(x=0, xanchor="left", font=dict(color=TINTA2, size=14))

fig.add_annotation(
    x="MUT3", y=0.42, showarrow=False, xanchor="center", row=2, col=1,
    text="<b>o pior mutante é o<br>mais bem avaliado</b>",
    font=dict(color=SUPERFICIE, size=12),
)
fig.add_annotation(
    x=0, y=-0.40, xref="paper", yref="paper", showarrow=False, xanchor="left", align="left",
    text=("<b>Pela lente nominal o poder é zero nos quatro:</b> o corte de §6.5 exige trajetória "
          "perfeita e P1 vale S2, então base e mutante reprovam igual — é o X33<br>"
          "medido onde ele mais dói. <b>E a lente rica não salva sozinha:</b> no MUT3, cortar o "
          "budget de 12 para 3 tira do agente a OPORTUNIDADE de disparar<br>"
          "falha, o conjunto de códigos encolhe, e o agente sabotado parece melhor. Vários "
          "códigos da taxonomia são proporcionais ao número de passos."),
    font=dict(color=TINTA2, size=11),
)
fg.exportar(fig, "fig08_ins9_mutantes", FIGURAS)
fig.show()

---
## 8. Os números da T28 em disco

`docs/resultados_h0.json` já existia com H0; ganha agora H2 e INS.9 ao lado. É o arquivo que o
README e os slides citam, para que nenhum número apareça neles digitado à mão.

In [13]:
resumo["h2"] = {
    d.metrica: {
        "delta": round(d.delta, 4),
        "ic95": [round(v, 4) for v in d.ic95],
        "n_pares": d.n_pares,
        "n_descartados": d.n_descartados,
        "p_bootstrap": round(d.p_bootstrap, 4),
        "veredito_estavel": d.veredito_estavel,
        "leitura": d.leitura,
    }
    for d in h2
}
resumo["ins9"] = {
    lente: {
        (d.variante or "agregada"): {
            "valor": round(d.valor, 4),
            "poder_util": round(d.poder_util, 4),
            "invertida": round(d.fracao_invertida, 4),
            "n_pares": d.n_pares,
        }
        for d in mut.tabela(pares_mut)
        if d.lente == lente
    }
    for lente in LENTE
}
(RAIZ / "docs" / "resultados_h0.json").write_text(
    json.dumps(resumo, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
print(json.dumps({"h2": resumo["h2"], "ins9": resumo["ins9"]["codigos"]},
                 indent=2, ensure_ascii=False))

{
  "h2": {
    "tool_f1": {
      "delta": 0.0206,
      "ic95": [
        -0.0212,
        0.0616
      ],
      "n_pares": 144,
      "n_descartados": 0,
      "p_bootstrap": 0.331,
      "veredito_estavel": true,
      "leitura": "cruza zero"
    },
    "tool_f1_liquido": {
      "delta": 0.0569,
      "ic95": [
        0.0076,
        0.107
      ],
      "n_pares": 144,
      "n_descartados": 0,
      "p_bootstrap": 0.0242,
      "veredito_estavel": true,
      "leitura": "não cruza zero"
    },
    "args_acc": {
      "delta": -0.0614,
      "ic95": [
        -0.1234,
        0.0003
      ],
      "n_pares": 124,
      "n_descartados": 21,
      "p_bootstrap": 0.0514,
      "veredito_estavel": false,
      "leitura": "no limiar — este n não decide"
    },
    "decisao_correta": {
      "delta": -0.1019,
      "ic95": [
        -0.2037,
        0.0
      ],
      "n_pares": 108,
      "n_descartados": 37,
      "p_bootstrap": 0.0696,
      "veredito_estavel": true,
      "leitura